In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.metrics import roc_curve, auc, roc_auc_score,precision_recall_curve, average_precision_score,precision_score,recall_score,f1_score, accuracy_score, classification_report, confusion_matrix, RocCurveDisplay
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.feature_selection import RFE
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedBaggingClassifier,EasyEnsembleClassifier,RUSBoostClassifier
np.random.seed(42)

### Disclaimer: As the datasets that we are handling have the missing values and categorical values already handled, we will therefore not implementing those sections from the papers we are modelling based on.

## Paper: 
#### https://www.researchgate.net/publication/365929175_Liver_Cirrhosis_Prediction_using_Machine_Learning_Approaches
### The methodology of the above paper in implementing the Random Forest model has been replicated below with our dataset.

In [2]:
#df = pd.read_csv("D:/IDEAS Project/data/ILPD_robust_scaled_features.csv")
df = pd.read_csv("D:/IDEAS Project/data/ILPD_robust_scaled_with_gmm_noise.csv")
df.head()

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Proteins,Albumin,Albumin_and_Globulin_Ratio,Result,Synthetic_Flag
0,0.80,1.0,-0.166667,-0.181818,-0.172131,-0.513514,-0.372470,0.142857,0.166667,-0.125,1,0
1,0.68,0.0,5.500000,4.727273,4.024590,0.783784,0.955466,0.642857,0.083333,-0.525,1,0
2,0.68,0.0,3.500000,3.454545,2.311475,0.675676,0.437247,0.285714,0.166667,-0.150,1,0
3,0.52,0.0,0.000000,0.090909,-0.213115,-0.567568,-0.340081,0.142857,0.250000,0.125,1,0
4,1.08,0.0,1.611111,1.545455,-0.106557,-0.216216,0.291498,0.500000,-0.583333,-1.375,1,0


In [3]:
target_col = "Result"
df[target_col] = df[target_col].replace({1: 1, 2: 0})
print("\nTarget value counts before SMOTE:")
print(df[target_col].value_counts())


Target value counts before SMOTE:
Result
1    487
0    197
Name: count, dtype: int64


In [4]:
#X = df.drop(columns=[target_col])
X=df.drop(columns=[target_col, "Synthetic_Flag"])
y = df[target_col]
smote = SMOTE(random_state=42,k_neighbors=3)
#smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)
print("\nTarget value counts after SMOTE:")
print(pd.Series(y_resampled).value_counts())


Target value counts after SMOTE:
Result
1    487
0    487
Name: count, dtype: int64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled,
    y_resampled,
    test_size=0.20,
    random_state=42,
    stratify=y_resampled
)
print("\nTraining set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


Training set shape: (779, 10)
Testing set shape: (195, 10)


In [6]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("\n===== RANDOM FOREST RESULTS =====")
print(classification_report(y_test, y_pred_rf))


===== RANDOM FOREST RESULTS =====
              precision    recall  f1-score   support

           0       0.77      0.90      0.83        98
           1       0.88      0.73      0.80        97

    accuracy                           0.82       195
   macro avg       0.82      0.81      0.81       195
weighted avg       0.82      0.82      0.81       195



In [7]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")

Accuracy : 0.8154
Precision: 0.8765
Recall   : 0.7320
F1 Score : 0.7978


#### We now add additional tuning on top of the paper's work

In [8]:
#run this tuning for gmm noise dataset only
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("\n===== RANDOM FOREST RESULTS =====")
print(classification_report(y_test, y_pred_rf))


===== RANDOM FOREST RESULTS =====
              precision    recall  f1-score   support

           0       0.80      0.89      0.84        98
           1       0.87      0.77      0.82        97

    accuracy                           0.83       195
   macro avg       0.84      0.83      0.83       195
weighted avg       0.83      0.83      0.83       195




## Paper: 
#### https://www.sciencedirect.com/science/article/pii/S2667305325000997
### The methodologies of the above paper in implementing HistGradientBoosting, RUSBoost and EasyEnsemble have been replicated below with our dataset.

In [9]:
# df = pd.read_csv("D:/IDEAS Project/data/ILPD_robust_scaled_features.csv")
# target_col = "Result"
# X = df.drop(columns=[target_col]).copy()
# y = df[target_col].copy()

df = pd.read_csv("D:/IDEAS Project/data/ILPD_robust_scaled_with_gmm_noise.csv")
target_col = "Result"
# Drop the extra metadata column from model input
X = df.drop(columns=[target_col, "Synthetic_Flag"]).copy()
y = df[target_col].copy()

In [10]:
y = y.map({1: 1, 2: 0})
print("\nClass distribution:")
print(y.value_counts())


Class distribution:
Result
1    487
0    197
Name: count, dtype: int64


In [11]:
X = df.drop(columns=[target_col])
y = df[target_col]

In [12]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
X_train_raw = X_train_raw.reset_index(drop=True)
X_test_raw = X_test_raw.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
print("\nTrain shape:", X_train_raw.shape)
print("Test shape:", X_test_raw.shape)
print("\nTrain class distribution:")
print(y_train.value_counts())
print("\nTest class distribution:")
print(y_test.value_counts())


Train shape: (547, 11)
Test shape: (137, 11)

Train class distribution:
Result
1    389
2    158
Name: count, dtype: int64

Test class distribution:
Result
1    98
2    39
Name: count, dtype: int64


In [13]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_raw, y_train)
X_train_smote = pd.DataFrame(X_train_smote, columns=X.columns)
y_train_smote = pd.Series(y_train_smote, name=target_col)
print("\nAfter SMOTE:")
print("X_train_smote shape:", X_train_smote.shape)
print("y_train_smote shape:", y_train_smote.shape)
print(y_train_smote.value_counts())


After SMOTE:
X_train_smote shape: (778, 11)
y_train_smote shape: (778,)
Result
1    389
2    389
Name: count, dtype: int64


In [14]:
rfe_estimator = LogisticRegression(max_iter=5000, random_state=42)
rfe = RFE(estimator=rfe_estimator, n_features_to_select=4)
rfe.fit(X_train_smote, y_train_smote)
selected_features = X_train_smote.columns[rfe.support_].tolist()
print("\nSelected Features by RFE:")
for feature in selected_features:
    print(feature)


Selected Features by RFE:
Age
Direct_Bilirubin
Alamine_Aminotransferase
Aspartate_Aminotransferase


In [15]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_smote[selected_features])
X_test_poly = poly.transform(X_test_raw[selected_features])
poly_feature_names = poly.get_feature_names_out(selected_features)
X_train_poly_df = pd.DataFrame(X_train_poly, columns=poly_feature_names)
X_test_poly_df = pd.DataFrame(X_test_poly, columns=poly_feature_names)
# combine original + polynomial features
X_train_final = pd.concat(
    [X_train_smote.reset_index(drop=True), X_train_poly_df.reset_index(drop=True)],
    axis=1
)
X_test_final = pd.concat(
    [X_test_raw.reset_index(drop=True), X_test_poly_df.reset_index(drop=True)],
    axis=1
)
print("\nFinal train shape:", X_train_final.shape)
print("Final test shape:", X_test_final.shape)


Final train shape: (778, 25)
Final test shape: (137, 25)


In [16]:
models = {
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
    "EasyEnsemble": EasyEnsembleClassifier(random_state=42),
    "RUSBoost": RUSBoostClassifier(random_state=42)
}

#### (Although the original paper emphasized a Quadratic Discriminant Analysis-based approach due to its ability to model class-specific covariance structures, it did not perform well on the ILPD dataset. This is primarily because ILPD is a binary and highly imbalanced dataset, where QDA tends to become overly conservative, resulting in very low recall despite high precision.

#### Therefore, while QDA is theoretically suitable for datasets with heterogeneous class distributions, it was not the most appropriate choice for the ILPD dataset, and imbalance-aware ensemble models were preferred for better practical performance.)

In [17]:
results = []
predictions = {}
probabilities = {}
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_final, y_train_smote)
    y_pred = model.predict(X_test_final)
    y_prob = model.predict_proba(X_test_final)[:, 1]
    predictions[name] = y_pred
    probabilities[name] = y_prob
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, y_prob),
        "PR AUC": average_precision_score(y_test, y_prob)
    })
results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False)
print("\n===== FINAL RESULTS =====")
print(results_df)


Training HistGradientBoosting...

Training EasyEnsemble...

Training RUSBoost...

===== FINAL RESULTS =====
                  Model  Accuracy  Precision    Recall  F1 Score   ROC AUC  \
0  HistGradientBoosting  0.678832   0.793478  0.744898  0.768421  0.769754   
1          EasyEnsemble  0.627737   0.821918  0.612245  0.701754  0.753140   
2              RUSBoost  0.627737   0.821918  0.612245  0.701754  0.753140   

     PR AUC  
0  0.576167  
1  0.590756  
2  0.590756  


## Paper: 
#### https://www.researchgate.net/publication/391697269_Enhancing_Liver_Cirrhosis_Diagnosis_Using_Machine_Learning_with_Explainable_AI_and_Cross-Validated_Hyperparameter_Tuning_Techniques
### The methodology of the above paper in implementing the tuned GradientBoosting model has been replicated below with our dataset.

In [18]:
#df = pd.read_csv("D:/IDEAS Project/data/ILPD_robust_scaled_features.csv")
df = pd.read_csv("D:/IDEAS Project/data/ILPD_robust_scaled_with_gmm_noise.csv")
df.head()

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Proteins,Albumin,Albumin_and_Globulin_Ratio,Result,Synthetic_Flag
0,0.80,1.0,-0.166667,-0.181818,-0.172131,-0.513514,-0.372470,0.142857,0.166667,-0.125,1,0
1,0.68,0.0,5.500000,4.727273,4.024590,0.783784,0.955466,0.642857,0.083333,-0.525,1,0
2,0.68,0.0,3.500000,3.454545,2.311475,0.675676,0.437247,0.285714,0.166667,-0.150,1,0
3,0.52,0.0,0.000000,0.090909,-0.213115,-0.567568,-0.340081,0.142857,0.250000,0.125,1,0
4,1.08,0.0,1.611111,1.545455,-0.106557,-0.216216,0.291498,0.500000,-0.583333,-1.375,1,0


In [19]:
target_col = "Result"
#X = df.drop(columns=[target_col])
X = df.drop(columns=[target_col, "Synthetic_Flag"])
y = df[target_col].replace({1: 1, 2: 0})
print("Class distribution before split:")
print(y.value_counts())

Class distribution before split:
Result
1    487
0    197
Name: count, dtype: int64


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [21]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("\nTraining class distribution before SMOTE:")
print(pd.Series(y_train).value_counts())
print("\nTraining class distribution after SMOTE:")
print(pd.Series(y_train_smote).value_counts())


Training class distribution before SMOTE:
Result
1    389
0    158
Name: count, dtype: int64

Training class distribution after SMOTE:
Result
1    389
0    389
Name: count, dtype: int64


In [22]:
gb_model = GradientBoostingClassifier(random_state=42)
gb_param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 5]
}

In [23]:
gb_grid = GridSearchCV(
    estimator=gb_model,
    param_grid=gb_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [24]:
gb_grid.fit(X_train_smote, y_train_smote)
best_gb = gb_grid.best_estimator_
print("\nBest Parameters:", gb_grid.best_params_)
print("Best CV Score:", gb_grid.best_score_)


Best Parameters: {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200}
Best CV Score: 0.8150537634408602


In [25]:
y_pred_gb = best_gb.predict(X_test)
print("\nGradient Boosting Performance:")
print(f"Accuracy : {accuracy_score(y_test, y_pred_gb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_gb, pos_label=1):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_gb, pos_label=1):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred_gb, pos_label=1):.4f}")


Gradient Boosting Performance:
Accuracy : 0.7080
Precision: 0.8021
Recall   : 0.7857
F1 Score : 0.7938


In [26]:
#!jupyter nbconvert --to html final_models.ipynb --output robust_scaled_outputs
!jupyter nbconvert --to html final_models.ipynb --output robust_scaled_gmm_noise_outputs

[NbConvertApp] Converting notebook final_models.ipynb to html
[NbConvertApp] Writing 348839 bytes to robust_scaled_gmm_noise_outputs.html
